In [6]:
import yaml
from app.datasets.loader import load_multiple_test_cases, load_test_cases
from app.datasets.validator import validate_dataset_schema
from app.datasets.validator import validate_dataset_schema
from app.client.rag_client import RAGClient
from app.tests.run_tests import run_tests

In [9]:
file_list = [
  './app/data/raw/test.xlsx',
]

test_config = {
  'GENERAL_TESTS': True,
  'TIMINGS': {'test': True, 'report': False},
  'TOKENS': {'test': True, 'report': False},
  'FOUNDRYS': {'test': True, 'report': False},
  'TRIAGE': {'test': True, 'report': False},
  'ROUTER': {'test': True, 'report': False},
  'GROUNDING': {'test': True, 'report': False},
  'SAVE_RESULTS': False,
  'PATH': './app/data/processed/reports/beto',
  
  'REFORMULATE': {'test': False, 'report': False}
}   

if file_list: 
  df = load_multiple_test_cases(file_list)
  df = validate_dataset_schema(df)

with open('./app/config/config.yaml', 'r') as file:
  config_data = yaml.load(file, Loader= yaml.FullLoader) 
  
client = RAGClient(config_data)
test_timestamps = {}

In [8]:
# TEST GENERALES
if test_config.get('GENERAL_TESTS', False):
  responses = client.query_batch(df['user_input'],df['reference'])
  save_responses_in_json, response_file_path = client.save_api_responses(responses)
  test_timestamps['general_tests'] = str(response_file_path).replace('\\', '/')

Processing queries: 100%|██████████| 15/15 [02:04<00:00,  8.33s/it]


In [ ]:
# import json

# response_file_path = './app/data/processed/outcomes/outcome_20260508-104150.json'
# with open(response_file_path, 'r', encoding='UTF-8') as f:
#   responses = json.load(f)
# test_timestamps['general_tests'] = 'outcome_20260508-104150.json'

In [13]:
if test_config.get('GENERAL_TESTS'):
  results, reports = run_tests(
    config = test_config, 
    data = responses, 
    df = df, 
    timestamp = test_timestamps
)

In [5]:
print(results)

{'timestamp': '20260508-101521', 'nodes': {'triage': {'positives': 14, 'total': 14, 'result': 100.0}, 'router': {'positives': 14, 'total': 14, 'result': 100.0}, 'grounding': {'positives': 10, 'total': 10, 'result': 100.0}}, 'timings': {'reformulate': {'prom': 1.522, 'med': 1.335, 'p90': 2.273, 'p95': 2.416, 'p99': 2.451, 'quantity': 14}, 'triage': {'prom': 1.36, 'med': 1.169, 'p90': 1.986, 'p95': 2.215, 'p99': 2.544, 'quantity': 14}, 'router': {'prom': 0.0, 'med': 0.0, 'p90': 0.0, 'p95': 0.0, 'p99': 0.0, 'quantity': 0}, 'ag_call': {'prom': 3.612, 'med': 3.367, 'p90': 4.159, 'p95': 5.308, 'p99': 6.227, 'quantity': 10}, 'personality': {'prom': 2.559, 'med': 2.472, 'p90': 3.617, 'p95': 3.633, 'p99': 3.646, 'quantity': 10}, 'grounding': {'prom': 1.621, 'med': 1.516, 'p90': 2.423, 'p95': 2.762, 'p99': 3.033, 'quantity': 10}, 'retriever': {'prom': 0.675, 'med': 0.672, 'p90': 0.796, 'p95': 0.904, 'p99': 0.99, 'quantity': 10}, 'ret_embeddings': {'prom': 0.374, 'med': 0.351, 'p90': 0.502, 'p95'